In [1]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../data/raw")

claims = pd.read_csv(DATA_DIR / "claims.csv")
policies = pd.read_csv(DATA_DIR / "policies.csv")
garages = pd.read_csv(DATA_DIR / "garages.csv")
adjusters = pd.read_csv(DATA_DIR / "adjusters.csv")

for name, frame in {
    "claims": claims,
    "policies": policies,
    "garages": garages,
    "adjusters": adjusters,
}.items():
    print(f"{name}: {frame.shape}")
    display(frame.head(3))

claims: (32396, 17)


,claim_id,policy_id,garage_id,adjuster_id,incident_type,incident_date,incident_hour,reported_date,claim_amount_xaf,police_report,witness_count,prior_claims_holder,vehicle_towed,investigation_opened,days_to_settle,amount_paid_xaf,fraud_flag
0,CLM-029155,PO910902,GR-001,AD-013,collision,2026-06-23,16,23/07/2026,421152,1,3,1,0,False,19.0,349582,NO
1,CLM-011041,PO904917,GR-061,AD-004,collision,2023-10-05,12,2023-10-07,1963058,1,1,1,TRUE,False,46.0,1957665,NO
2,CLM-005039,PO909465,GR-033,AD-039,rollover,2025-06-14,10,2025-06-28,2491278,yes,1,0,yes,False,2.0,2133099,NO


policies: (26000, 10)


,policy_id,holder_id,region,vehicle_make,vehicle_year,cover_type,sum_insured_xaf,annual_premium_xaf,policy_start,payment_frequency
0,PO900001,HL416734,Bertoua,Isuzu,2013,third party,3200000,139728,2025-02-18,quarterly
1,PO900002,HL415591,Buea,Hyundai,1999,third party fire and theft,10250000,554017,2024-11-25,quarterly
2,PO900003,HL402440,Douala,Mitsubishi,2013,third party,6300000,227476,2025-05-04,annual


garages: (70, 6)


,garage_id,garage_name,town,registered_year,bay_count,approved
0,GR-001,Bon Depart Auto 1,Douala,2017,6,0
1,GR-002,Bahia Auto 2,Yaounde,2014,7,yes
2,GR-003,Tonnerre Auto 3,Maroua,1998,2,0


adjusters: (45, 4)


,adjuster_id,region,hired_year,caseload_band
0,AD-001,West,2011,medium
1,AD-002,Littoral,2020,low
2,AD-003,North,2015,high


In [2]:
print("Fraud labels:")
display(claims["fraud_flag"].value_counts(dropna=False))
print(f"\nFraud prevalence: {(claims['fraud_flag'] == 'YES').mean():.2%}")

print("\nExact duplicate claim rows:", claims.duplicated().sum())

print("\nMissing values:")
display(claims.isna().sum().sort_values(ascending=False))

for column in ["incident_date", "reported_date"]:
    print(f"\n{column} format examples and counts:")
    display(
        claims[column]
        .astype("string")
        .str.replace(r"\d", "D", regex=True)
        .value_counts()
        .head(10)
    )

for column in ["police_report", "vehicle_towed", "investigation_opened"]:
    print(f"\n{column} values:")
    display(claims[column].value_counts(dropna=False))

Fraud labels:


fraud_flag
NO     31410
YES      986
Name: count, dtype: int64


Fraud prevalence: 3.04%

Exact duplicate claim rows: 220

Missing values:


days_to_settle          551
claim_id                  0
policy_id                 0
adjuster_id               0
garage_id                 0
incident_date             0
incident_hour             0
reported_date             0
incident_type             0
claim_amount_xaf          0
police_report             0
prior_claims_holder       0
witness_count             0
vehicle_towed             0
investigation_opened      0
amount_paid_xaf           0
fraud_flag                0
dtype: int64


incident_date format examples and counts:


...

police_report
yes      5289
1        5281
True     5193
TRUE     5156
0        2959
FALSE    2904
no       2820
False    2794
Name: count, dtype: int64


vehicle_towed values:


vehicle_towed
False    5456
0        5432
no       5402
TRUE     5392
1        5390
yes      5324
Name: count, dtype: int64


investigation_opened values:


investigation_opened
False    30261
True      2135
Name: count, dtype: int64

In [3]:
def parse_claim_date(raw_dates: pd.Series) -> pd.Series:
    raw = raw_dates.astype("string").str.strip()
    parsed = pd.Series(pd.NaT, index=raw.index, dtype="datetime64[ns]")

    iso = raw.str.fullmatch(r"\d{4}-\d{2}-\d{2}", na=False)
    slash = raw.str.fullmatch(r"\d{2}/\d{2}/\d{4}", na=False)
    written = raw.str.fullmatch(r"\d{2} [A-Za-z]{3} \d{4}", na=False)

    parsed.loc[iso] = pd.to_datetime(raw.loc[iso], format="%Y-%m-%d")
    parsed.loc[slash] = pd.to_datetime(raw.loc[slash], format="%d/%m/%Y")
    parsed.loc[written] = pd.to_datetime(raw.loc[written], format="%d %b %Y")

    unknown = raw[~(iso | slash | written)]
    if not unknown.empty:
        raise ValueError(f"Unexpected date formats: {unknown.unique()[:5]}")

    return parsed


claims["incident_at"] = parse_claim_date(claims["incident_date"])
claims["reported_at"] = parse_claim_date(claims["reported_date"])

print("Incident period:", claims["incident_at"].min(), "to", claims["incident_at"].max())
print("Reported period:", claims["reported_at"].min(), "to", claims["reported_at"].max())
print("Claims reported before their incident:", (claims["reported_at"] < claims["incident_at"]).sum())

Incident period: 2023-06-07 00:00:00 to 2026-06-30 00:00:00
Reported period: 2023-06-07 00:00:00 to 2026-08-14 00:00:00
Claims reported before their incident: 0


In [4]:
claims_clean = claims.drop_duplicates().copy()
claims_clean["target"] = (claims_clean["fraud_flag"] == "YES").astype(int)

latest_reported_at = claims_clean["reported_at"].max()
final_holdout_start = latest_reported_at - pd.DateOffset(months=6)
development_test_start = final_holdout_start - pd.DateOffset(months=6)

print("Latest reported date:", latest_reported_at.date())
print("Development test starts:", development_test_start.date())
print("Final holdout starts:", final_holdout_start.date())

for name, mask in {
    "development training": claims_clean["reported_at"] < development_test_start,
    "development temporal test": (
        (claims_clean["reported_at"] >= development_test_start)
        & (claims_clean["reported_at"] < final_holdout_start)
    ),
    "final six-month holdout": claims_clean["reported_at"] >= final_holdout_start,
}.items():
    subset = claims_clean.loc[mask]
    print(f"{name}: {len(subset):,} claims; fraud prevalence = {subset['target'].mean():.2%}")

garage_summary = (
    claims_clean.groupby("garage_id")["target"]
    .agg(claims="size", fraud_rate="mean")
    .sort_values("fraud_rate", ascending=False)
)

display(garage_summary.head(10))
display(garage_summary["fraud_rate"].describe())

Latest reported date: 2026-08-14
Development test starts: 2025-08-14
Final holdout starts: 2026-02-14
development training: 16,933 claims; fraud prevalence = 2.35%
development temporal test: 8,564 claims; fraud prevalence = 3.71%
final six-month holdout: 6,679 claims; fraud prevalence = 3.95%


,claims,fraud_rate
garage_id,,
GR-006,479,0.225470
GR-066,426,0.173709
GR-028,440,0.159091
GR-022,455,0.158242
GR-024,458,0.152838
GR-019,413,0.147700
GR-036,427,0.126464
GR-012,495,0.125253
GR-068,447,0.098434


count    70.000000
mean      0.030873
std       0.050483
min       0.000000
25%       0.006319
50%       0.010364
75%       0.018412
max       0.225470
Name: fraud_rate, dtype: float64